# 🏪 Notebook 1: Setup y Primer Agente con Strands

## Objetivos
- Configurar el entorno de desarrollo para el curso
- Entender la arquitectura básica de Strands Agents
- Crear un agente de customer service conectado a Amazon Bedrock
- Definir herramientas (tools) para el agente
- Ejecutar consultas y observar el comportamiento

## Contexto
TechShop es una tienda online de electrónica. Vamos a construir un agente de IA que atienda a sus clientes.
El agente tiene dos herramientas: buscar en el catálogo de productos y consultar las preguntas frecuentes.

> **Pregunta para pensar:** ¿Cómo sabremos si el agente funciona correctamente una vez que lo tengamos en producción?

## 1. Instalación de dependencias

Instalamos las librerías necesarias para crear agentes con Strands y conectarlos a Amazon Bedrock.

In [5]:
# Verificar dependencias (instaladas via: cd notebooks/ && uv sync)
import importlib.metadata

import boto3

print(f"✅ strands {importlib.metadata.version('strands-agents')}")
print(f"✅ boto3 {boto3.__version__}")

✅ strands 1.27.0
✅ boto3 1.42.53


## 2. Configuración del entorno

Cargamos las variables de entorno desde el archivo `.env`. 
Si aún no lo tienes, copia `.env.example` a `.env` y rellena con tus credenciales AWS.

```bash
cp ../.env.example .env
# Edita .env con tus credenciales
```

In [6]:
import os
from dotenv import load_dotenv

# Carga variables de entorno desde .env
load_dotenv()

# Verificar que las variables AWS están configuradas
assert os.getenv("AWS_ACCESS_KEY_ID"), "❌ Falta AWS_ACCESS_KEY_ID en .env"
assert os.getenv("AWS_SECRET_ACCESS_KEY"), "❌ Falta AWS_SECRET_ACCESS_KEY en .env"

print(f"✅ Región AWS: {os.getenv('AWS_REGION', 'us-east-1')}")
print(f"✅ Modelo: {os.getenv('MODEL_ID', 'us.anthropic.claude-haiku-4-5-20251001-v1:0')}")

✅ Región AWS: us-east-1
✅ Modelo: us.anthropic.claude-haiku-4-5-20251001-v1:0


## 3. Verificar conexión a Amazon Bedrock

Antes de crear el agente, verificamos que podemos conectarnos a Bedrock.

In [8]:
bedrock = boto3.client(
    "bedrock-runtime",
    region_name=os.getenv("AWS_REGION", "us-east-1")
)
print("✅ Conexión a Bedrock Runtime establecida")

✅ Conexión a Bedrock Runtime establecida


## 4. Explorar los datos de TechShop

El agente trabaja con dos fuentes de datos locales:
- **Catálogo de productos** (`catalog.json`): productos con nombre, precio, stock y descripción
- **Preguntas frecuentes** (`faqs.json`): políticas de envío, devoluciones, garantías, etc.

In [12]:
import json
from pathlib import Path

# Los datos están en el repositorio
DATA_DIR = Path("../../src/techshop_agent/data")

FAQS_FILE = DATA_DIR / "faqs.json"
CATALOG_FILE = DATA_DIR / "catalog.json"

N_ELEMENTS = 10

# Cargar catálogo
with CATALOG_FILE.open(encoding="utf-8") as f:
    catalog = json.load(f)

print(f"Productos en catálogo: {len(catalog)}")
print(f"   Categorías: {sorted({p['categoria'] for p in catalog})}")
print()
for p in catalog[:N_ELEMENTS]:
    print(f"   • {p['nombre']:20s} | {p['precio']:>8.2f}€ | Stock: {p['stock']}")

Productos en catálogo: 16
   Categorías: ['accesorios', 'almacenamiento', 'audio', 'fotografia', 'gaming', 'perifericos', 'portatiles', 'redes', 'smartphones', 'tablets', 'wearables']

   • ProBook X1           |  1199.99€ | Stock: 12
   • NanoBook Air 13      |   999.50€ | Stock: 8
   • VoltPhone S          |   749.00€ | Stock: 20
   • VoltPhone Mini       |   599.00€ | Stock: 15
   • AuralPods Pro        |   199.99€ | Stock: 35
   • AuralPods Lite       |    89.90€ | Stock: 50
   • OrbitWatch 2         |   249.00€ | Stock: 18
   • PixelCam Z           |   549.00€ | Stock: 9
   • HomeMesh AX          |   179.00€ | Stock: 22
   • KeyMaster TKL        |   129.00€ | Stock: 27


In [ ]:
# Cargar FAQs
with FAQS_FILE.open(encoding="utf-8") as f:
    faqs = json.load(f)

print(f"Preguntas frecuentes: {len(faqs)}")
print(f"   Temas: {sorted({faq['tema'] for faq in faqs})}")
print()
for faq in faqs[:N_ELEMENTS]:
    print(f"   [{faq['tem0a']:12s}] {faq['pregunta']}")

Preguntas frecuentes: 11
   Temas: ['devoluciones', 'envios', 'garantias', 'horarios', 'pagos', 'soporte']

   [devoluciones] ¿Cuál es la política de reembolso?
   [envios      ] ¿Cuánto tarda el envío estándar?
   [envios      ] ¿Ofrecen envío exprés?
   [garantias   ] ¿Qué cubre la garantía de productos?
   [pagos       ] ¿Cuáles son los métodos de pago disponibles?
   [pagos       ] ¿Puedo pagar en cuotas?
   [horarios    ] ¿Cuál es el horario de atención al cliente?
   [soporte     ] ¿Cómo solicito soporte técnico?
   [devoluciones] ¿Qué pasa si mi pedido llega dañado?
   [envios      ] ¿Realizan envíos internacionales?


## 5. Crear el modelo Bedrock

Strands Agents usa `BedrockModel` para conectarse a los modelos de Amazon Bedrock a través de la API Converse.
Esto nos da acceso a Claude, Nova, y otros modelos disponibles en la región.

In [19]:
from strands.models import BedrockModel

model = BedrockModel(
    model_id=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
    temperature=0.3,
    max_tokens=1024,
)

print(f"✅ Modelo configurado: {model.config.get('model_id', 'Not Set')}")

✅ Modelo configurado: us.anthropic.claude-haiku-4-5-20251001-v1:0


## 6. Definir las herramientas (Tools) del agente

Las herramientas son funciones Python decoradas con `@tool` que el agente puede invocar.
El agente decide cuándo y cómo usarlas basándose en la consulta del usuario.

> **Concepto clave:** El agente es un LLM + herramientas + system prompt. El LLM decide la estrategia, las herramientas ejecutan acciones concretas.

In [24]:
from strands import tool


@tool
def search_catalog(query: str) -> str:
    """Busca productos en el catálogo de TechShop por nombre o descripción.
    Usa esta herramienta cuando el usuario pregunte sobre productos, precios o disponibilidad."""
    catalog_path = DATA_DIR / "catalog.json"
    catalog = json.loads(catalog_path.read_text(encoding="utf-8"))

    query_lower = query.lower()
    matches = []
    for product in catalog:
        searchable = f"{product['nombre']} {product['descripcion']}".lower()
        if query_lower in searchable:
            matches.append({
                "nombre": product["nombre"],
                "precio": product["precio"],
                "stock": product["stock"],
                "descripcion": product["descripcion"],
            })

    if not matches:
        return f"No se encontraron productos para: {query}"
    return json.dumps(matches, ensure_ascii=False, indent=2)

print("✅ Herramienta search_catalog definida")

✅ Herramienta search_catalog definida


In [25]:
@tool
def get_faq_answer(question: str) -> str:
    """Busca respuestas en las preguntas frecuentes de TechShop.
    Usa esta herramienta para consultas sobre envíos, devoluciones, garantías, pagos u horarios."""
    faqs_path = DATA_DIR / "faqs.json"
    faqs = json.loads(faqs_path.read_text(encoding="utf-8"))

    # Búsqueda simple por palabras clave
    stopwords = {"el", "la", "los", "las", "de", "del", "y", "o", "un", "una", "que", "en"}
    words = [w for w in question.lower().split() if w not in stopwords]

    for faq in faqs:
        searchable = f"{faq['pregunta']} {faq['respuesta']}".lower()
        if any(word in searchable for word in words):
            return json.dumps(faq, ensure_ascii=False, indent=2)

    return f"No se encontró información sobre: {question}"

print("✅ Herramienta get_faq_answer definida")

✅ Herramienta get_faq_answer definida


## 7. Crear el agente TechShop

Ahora ensamblamos todo: modelo + herramientas + system prompt = agente.

El **system prompt** define la personalidad y las reglas del agente.

In [22]:
from strands import Agent

SYSTEM_PROMPT = """Eres un asistente de atención al cliente para TechShop, una tienda online de electrónica.

Tu trabajo es ayudar a los clientes con:
- Consultas sobre productos (precios, disponibilidad, especificaciones)
- Preguntas frecuentes (envíos, devoluciones, garantías, pagos, horarios)

Reglas:
- SIEMPRE usa las herramientas disponibles para buscar información antes de responder
- Si no encuentras la información, dilo honestamente
- Responde en español, de forma concisa y profesional
- No inventes información sobre productos o políticas
"""

agent = Agent(
    model=model,
    tools=[search_catalog, get_faq_answer],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agente TechShop creado y listo")

✅ Agente TechShop creado y listo


## 8. ¡Probemos el agente!

Vamos a lanzar diferentes tipos de consultas para ver cómo se comporta.

In [26]:
# Consulta sobre productos
print("=" * 70)
print("CONSULTA 1: Productos")
print("=" * 70)
response = agent("¿Qué portátiles tenéis disponibles?")
print(response)

CONSULTA 1: Productos

Tool #2: search_catalog

Tool #3: search_catalog

Tool #4: search_catalog
Lo siento, pero no tenemos portátiles disponibles en nuestro catálogo actualmente. He buscado con varios términos (portátiles, laptop, notebook, computadora portátil) y no hay resultados.

¿Hay algún otro producto electrónico que te interese? Puedo ayudarte a buscar:
- Tablets
- Monitores
- Teclados y ratones
- Accesorios informáticos
- Otros dispositivos electrónicos

¿Hay algo más en lo que pueda asistirte?Lo siento, pero no tenemos portátiles disponibles en nuestro catálogo actualmente. He buscado con varios términos (portátiles, laptop, notebook, computadora portátil) y no hay resultados.

¿Hay algún otro producto electrónico que te interese? Puedo ayudarte a buscar:
- Tablets
- Monitores
- Teclados y ratones
- Accesorios informáticos
- Otros dispositivos electrónicos

¿Hay algo más en lo que pueda asistirte?



In [27]:
# Consulta sobre FAQs
print("=" * 70)
print("CONSULTA 2: Políticas de la tienda")
print("=" * 70)
response = agent("¿Cuál es la política de devoluciones?")
print(response)

CONSULTA 2: Políticas de la tienda

Tool #5: get_faq_answer
Según nuestra política de devoluciones:

**Aceptamos solicitudes de reembolso dentro de los 30 días posteriores a la compra con comprobante válido.**

Si tienes alguna duda adicional sobre devoluciones o necesitas más información sobre otras políticas (envíos, garantías, pagos), no dudes en preguntar.Según nuestra política de devoluciones:

**Aceptamos solicitudes de reembolso dentro de los 30 días posteriores a la compra con comprobante válido.**

Si tienes alguna duda adicional sobre devoluciones o necesitas más información sobre otras políticas (envíos, garantías, pagos), no dudes en preguntar.



In [ ]:
# Consulta fuera de ámbito
print("=" * 70)
print("CONSULTA 3: Fuera de ámbito")
print("=" * 70)
response = agent("¿Cuál es la capital de Francia?")
print(response)

In [ ]:
# Consulta ambigua
print("=" * 70)
print("CONSULTA 4: Búsqueda que puede fallar")
print("=" * 70)
response = agent("Necesito algo para escuchar música mientras corro")
print(response)

## 10. Bonus: el mismo agente como paquete Python

Todo lo que acabamos de construir paso a paso ya existe empaquetado en `src/techshop_agent/`.
Desde el **Notebook 2 en adelante**, usaremos el paquete directamente:

```python
from techshop_agent import create_agent
agent = create_agent()
```

Esto nos permite centrarnos en **operacionalizar** el agente (observabilidad, prompts, evaluación, guardrails) sin redefinirlo cada vez.

In [28]:
# Importar el agente desde el paquete (funciona igual que el de arriba)
from techshop_agent import create_agent

agent_pkg = create_agent()
response = agent_pkg("¿Cuál es vuestra política de devoluciones?")
print(str(response)[:300])


Tool #1: get_faq_answer
¡Hola! Te comparto nuestra política de devoluciones en TechShop:

📋 **Puntos clave:**

- **Plazo de reembolso:** Aceptamos solicitudes de reembolso dentro de **30 días posteriores a la compra** con comprobante válido.

- **Reportar daños:** Si el producto llega dañado, debes reportarlo dentro de las **primeras 48 horas** con fotos del paquete y del producto.

- **Cambios de producto:** El cambio se gestiona dentro de **30 días**, sujeto a disponibilidad de stock y estado del artículo.

¿Hay algo más específico sobre devoluciones que necesites saber?¡Hola! Te comparto nuestra política de devoluciones en TechShop:

📋 **Puntos clave:**

- **Plazo de reembolso:** Aceptamos solicitudes de reembolso dentro de **30 días posteriores a la compra** con comprobante válido.

- **Reportar daños:** Si el producto llega dañado, debes reportarlo dentro de las


## 9. Reflexión: ¿Qué no sabemos?

El agente "funciona", pero hay preguntas que no podemos responder solo mirando el output:

| Pregunta | ¿Podemos responderla? |
|----------|----------------------|
| ¿Llamó a la herramienta correcta? | ❌ No lo vemos |
| ¿Cuántos tokens consumió? | ❌ No lo sabemos |
| ¿Cuánto tardó cada llamada? | ❌ No lo medimos |
| ¿Inventó información? | ❌ Solo si lo leemos manualmente |
| ¿Cuánto nos costó? | ❌ Sin dato |

> **Este es el problema que resuelve la observabilidad.** En el siguiente notebook, instrumentaremos el agente con Langfuse para responder todas estas preguntas.

---

## Siguiente: [Notebook 2 - Observabilidad con Langfuse](../day_1/02_observability.ipynb)